In [ ]:
#K means clustering is an unsupervised algorithm. Accuracy is vvv bad. 

In [24]:
import pandas as pd
import numpy as np
from tqdm import tqdm

np.random.seed(1)

data = pd.read_csv('data/diabetes_prediction_standardized.csv')
x = data.drop('diabetes', axis=1).to_numpy()
y = data['diabetes']

print(x.shape)

train = 5000
test = 7000

x_train = x[0:train,]

x_test = x[train:test,]
y_test = y[train:test,]

(96146, 12)


In [22]:
class KMeans():
    def __init__(self, N, dim):
        self.N = N
        self.centres = []
        for i in range(self.N):
            self.centres.append(np.random.rand(dim))
        self.label_mapping = {}

    def closestCluster(self,x,k):
        return np.argmin([np.linalg.norm(x - k[i]) for i in range(self.N)])

    def fit(self,x,y,epoch=100):
        for i in tqdm(range(epoch), desc = 'processing...'):
            current_cluster = np.array([self.closestCluster(x[i],self.centres) for i in range(x.shape[0])])
            self.centres = [x[current_cluster == i].mean(axis = 0) for i in range(self.N)]
        self.align_labels(x,y)

    def predict(self, x):      
        return np.array([self.closestCluster(x[j],self.centres) for j in range(x.shape[0])])

    def align_labels(self, x_train, y_train):
        train_preds = np.array(
            [self.closestCluster(x_train[i], self.centres) for i in range(x_train.shape[0])]
        )

        for i in range(self.N):
            true_labels_in_cluster = y_train[train_preds == i]
            if len(true_labels_in_cluster) > 0:
                most_frequent_label = np.bincount(true_labels_in_cluster).argmax()
                self.label_mapping[i] = most_frequent_label
            else:
                self.label_mapping[i] = i

In [23]:
kmeans = KMeans(2, x.shape[1])
kmeans.fit(x,y)
y_pred = kmeans.predict(x_test)
print(f'Accuracy: {(sum(y_test == y_pred)/y_pred.shape[0]) * 100}%')

processing...: 100%|██████████| 100/100 [02:04<00:00,  1.25s/it]


Accuracy: 49.85%
